###Env to use: birdnet

In [82]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf
import os
from pathlib import Path
from sklearn.metrics import average_precision_score

In [63]:
df_combined = pd.read_csv("df_comb_date_testcase3.csv")
df_combined.head(2)

,selec,filename,recording.id,date,target,fold
0,5098,5098_0,15646,20200629,0,1
1,8019,NaN,15629,20200629,1,1


In [65]:
df_combined.loc[df_combined["target"] == 1, "filename"] = (
    df_combined.loc[df_combined["target"] == 1, "selec"].astype(str)
)

Predict for positive segments first

In [66]:
# np.save('holderried_predictions.npy', predictions)
preds = np.load('holderried_predictions_selections.npy', allow_pickle=True)
preds_pos = preds.item()

In [67]:
df_pos = pd.DataFrame(preds_pos.items(),columns=["filename", "prediction"])

In [68]:
len(df_pos)

2410

In [69]:
# Create selec from filename
df_pos["selec"] = (
    df_pos["filename"]
    .str.replace(".wav", "", regex=False)
    .astype(int)
)

# Merge with fold information
df_pos = df_pos.merge(
    df_combined[df_combined.target==1],
    on="selec",
    how="left"
)

df_pos = df_pos.dropna()

print(df_pos.head())

   filename_x  prediction  selec filename_y  recording.id        date  target  \
0      10.wav    0.995466     10         10       11600.0  20200611.0     1.0   
10   1010.wav    0.284779   1010       1010       16819.0  20200630.0     1.0   
11   1011.wav    0.997902   1011       1011       16819.0  20200630.0     1.0   
12   1012.wav    0.656370   1012       1012       16819.0  20200630.0     1.0   
13   1013.wav    0.841130   1013       1013       16819.0  20200630.0     1.0   

    fold  
0    0.0  
10   3.0  
11   3.0  
12   3.0  
13   3.0  


Load Negative predictions

In [70]:
preds = np.load('holderried_preds_testdata3_neg.npy', allow_pickle=True)
preds_neg = preds.item()

In [71]:
df_neg = pd.DataFrame(preds_neg.items(),columns=["filename", "prediction"])

In [72]:
df_neg.head(2)

,filename,prediction
0,1116_0.wav,0.010087
1,1116_3.wav,0.016139


In [73]:
df_neg["filename"] = (
    df_neg["filename"]
    .str.replace(".wav", "", regex=False)
)

In [74]:
# Merge with fold information
df_neg = df_neg.merge(
    df_combined[df_combined.target==0],
    on="filename",
    how="left"
)

#df_pos = df_pos.dropna()

print(df_neg.head())

  filename  prediction  selec  recording.id      date  target  fold
0   1116_0    0.010087   1116         19145  20200619       0     2
1   1116_3    0.016139   1116         19145  20200619       0     2
2   1116_6    0.000000   1116         19145  20200619       0     2
3   5054_0    0.000000   5054         15603  20200627       0     1
4   5054_3    0.000000   5054         15603  20200627       0     1


In [75]:
len(df_pos)

513

In [76]:
df_all = pd.concat([df_pos, df_neg], ignore_index=True)

In [77]:
len(df_all)

801

In [78]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Combine positive and negative predictions
df_all = pd.concat([df_pos, df_neg], ignore_index=True)

# Make sure these are numeric
df_all["target"] = df_all["target"].astype(int)
df_all["fold"] = df_all["fold"].astype(int)

threshold = 0.5

results = []
confusion_matrices = []

for fold in sorted(df_all["fold"].unique()):

    df_fold = df_all[df_all["fold"] == fold]

    y_true = df_fold["target"].values
    y_score = df_fold["prediction"].values
    y_pred = (y_score >= threshold).astype(int)

    # Metrics
    ap = average_precision_score(y_true, y_score)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_score)

    cm = confusion_matrix(y_true, y_pred)
    confusion_matrices.append(cm)

    results.append({
        "fold": fold,
        "n_samples": len(df_fold),
        "n_positive": (y_true == 1).sum(),
        "n_negative": (y_true == 0).sum(),
        "average_precision": ap,
        "roc_auc": auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1],
    })

results_df = pd.DataFrame(results)

print(results_df)

   fold  n_samples  n_positive  n_negative  average_precision   roc_auc  \
0     0        204         151          53           0.892605  0.779333   
1     1        171          92          79           0.616350  0.627270   
2     2        205         177          28           0.965457  0.849274   
3     3        221          93         128           0.647648  0.700101   

   accuracy  precision    recall        f1  TN  FP  FN   TP  
0  0.578431   0.901235  0.483444  0.629310  45   8  78   73  
1  0.596491   0.657534  0.521739  0.581818  54  25  44   48  
2  0.663415   0.965517  0.632768  0.764505  24   4  65  112  
3  0.660633   0.593750  0.612903  0.603175  89  39  36   57  


In [79]:
metrics = [
    "average_precision",
    "roc_auc",
    "accuracy",
    "precision",
    "recall",
    "f1",
]

for metric in metrics:
    mean = results_df[metric].mean()
    std = results_df[metric].std()
    print(f"{metric}: {mean:.4f} ± {std:.4f}")

average_precision: 0.7805 ± 0.1745
roc_auc: 0.7390 ± 0.0962
accuracy: 0.6247 ± 0.0437
precision: 0.7795 ± 0.1815
recall: 0.5627 ± 0.0716
f1: 0.6447 ± 0.0822
